# Solo

Publication: https://www.cell.com/cell-systems/fulltext/S2405-4712(20)30195-2  
GitHub: https://github.com/calico/solo/blob/master/README.md  

In [ ]:
import scvi
import scanpy as sc

import pandas as pd

import os
import subprocess

import seaborn as sns

In [ ]:
sc.set_figure_params(figsize=(5, 5))

In [ ]:
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [ ]:
scvi.settings.num_threads=48

# Files and directories 

In [ ]:
adata_file = '../reference/scRNAseq/data/reynolds_et_al/data/human_adult_raw.h5ad'

# Import anndata 

In [ ]:
adata = sc.read_h5ad(adata_file)

In [ ]:
adata = adata[adata.obs['nCount_RNA']>=1000]
adata = adata[adata.obs['pMT_RNA']<=15]

# Run SCVI 

In [ ]:
def scvi_workflow(adata, sample_id):
    
    # Subset data 
    adata=adata[adata.obs['sample_id']==sample_id].copy()
    sc.pp.filter_genes(adata, min_counts=10)
    
    # Highly variable genes 
    sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=2000, subset=True)
    
    # Setup 
    scvi.model.SCVI.setup_anndata(adata)
    
    # Model 
    model=scvi.model.SCVI(adata, gene_likelihood='nb')
    
    # Train 
    model.train()
    
    # Update adata with scvi results  
    adata.layers['data']=model.get_normalized_expression(library_size=10e4)
    adata.obsm['latent']=model.get_latent_representation()
    
    # Dim reduction and clustering 
    sc.pp.neighbors(adata, n_neighbors=15, use_rep='latent')
    sc.tl.leiden(adata)
    sc.tl.umap(adata)
    
    # Save 
    model.save('data/object/reynolds_et_al_2021/solo/'+sample_id+'/'+'model/', overwrite=True)
    adata.write('data/object/reynolds_et_al_2021/solo/'+sample_id+'/'+'adata.h5ad')

In [ ]:
# [scvi_workflow(adata, sample_id) for sample_id in adata.obs['sample_id'].cat.categories]

# Run solo 

In [ ]:
def solo_workflow(sample_id):
    
    print(sample_id)
    
    # Load and prepare adata
    adata = sc.read_h5ad('data/object/reynolds_et_al_2021/solo/'+sample_id+'/'+'adata.h5ad')
    
    # Load scvi model  
    model = scvi.model.SCVI.load('data/object/reynolds_et_al_2021/solo/'+sample_id+'/'+'model/', adata)
    
    # Register solo model 
    solo = scvi.external.SOLO.from_scvi_model(model)
    solo.train()
    
    # Get doublet score and labels 
    prob = solo.predict(soft=True, include_simulated_doublets=False)
    label = solo.predict(soft=False, include_simulated_doublets=False)
    
    prob.index = prob.index.str[:-2]
    label.index = label.index.str[:-2]
    
    adata.obs['solo_pred_doublet'] = prob['doublet']
    adata.obs['solo_pred_singlet'] = prob['singlet']
    adata.obs['solo_label'] = label
    
    # Store results 
    solo.save('data/object/reynolds_et_al_2021/solo/'+sample_id+'/'+'model_solo/', save_anndata=True, overwrite=True)
    adata.write('data/object/reynolds_et_al_2021/solo/'+sample_id+'/'+'adata.h5ad')

In [ ]:
# [solo_workflow(sample_id) for sample_id in adata.obs['sample_id'].cat.categories]

# Plot UMAP

In [ ]:
def umap_plot(sample_id):
    
    # Print sample name 
    print(sample_id)
    
    # Load and prepare adata
    adata = sc.read_h5ad('data/object/reynolds_et_al_2021/solo/'+sample_id+'/'+'adata.h5ad')
    
    # Plot umap 
    sc.pl.umap(adata, color=['leiden', 'solo_pred_doublet', 'solo_pred_singlet', 'solo_label'], frameon=False, ncols=5, wspace=0.25)

In [ ]:
catch = [umap_plot(sample_id) for sample_id in adata.obs['sample_id'].cat.categories]

# MAD cluster analysis

## Combine SOLO results

In [ ]:
solo_pred = []

for sample_id in adata.obs['sample_id'].cat.categories:
    adata = sc.read_h5ad('data/object/reynolds_et_al_2021/solo/'+sample_id+'/'+'adata.h5ad')
    solo_pred.append(adata.obs[['solo_pred_doublet', 'solo_pred_singlet', 'solo_label']])

solo_pred = pd.concat(solo_pred, axis=0, ignore_index=False)

In [ ]:
adata = sc.read_h5ad(adata_file)
adata = adata[adata.obs['nCount_RNA']>=1500]
adata = adata[adata.obs['pMT_RNA']<=15]
sc.pp.filter_genes(adata, min_counts=10)

In [ ]:
adata.obs = adata.obs.merge(solo_pred, left_index=True, right_index=True)

## Build scvi model on combined data

In [ ]:
scvi.model.SCVI.setup_anndata(adata, categorical_covariate_keys=['sample_id'])

model = scvi.model.SCVI(
    adata, 
    n_latent=10, 
    n_hidden=128, 
    n_layers=2, 
    gene_likelihood='nb'
)

model.train()
model.save('data/object/reynolds_et_al_2021/solo/COMB/model', overwrite=True)

In [ ]:
# scvi normalized counts 
adata.obsm['latent'] = model.get_latent_representation()

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, use_rep='latent')
sc.tl.leiden(adata, resolution=1)
sc.tl.umap(adata)

## MAD score 

In [ ]:
meta_file = 'mad_tmp.csv'
adata.obs.to_csv(meta_file)

In [ ]:
# Define command and arguments
command = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.1.0-FD20220623HSCT/bin/Rscript'
path2script = '/research/peer/fdeckert/FD20220623HSCT/script/1_pre_processing/reynolds_et_al_2021/solo/mad.R'

# Variable number of args in a list
args = meta_file
# Build subprocess command
cmd = [command, path2script, args]
# check_output will run the command and store to result
subprocess.run(cmd, universal_newlines=True)

In [ ]:
meta = pd.read_csv(meta_file, index_col=0)
adata.obs['solo_mad_label'] = meta['solo_mad_label']

In [ ]:
adata.obs['solo_mad_label']

In [ ]:
os.remove(meta_file)

# UMAP

In [ ]:
sc.pl.umap(adata, color=['leiden', 'donor_id', 'solo_pred_doublet', 'solo_pred_singlet', 'solo_label','solo_mad_label'], frameon=False, ncols=3, wspace=0.5)

# Save results

In [ ]:
adata.obs[['solo_pred_doublet', 'solo_pred_singlet', 'solo_label', 'solo_mad_label']].to_csv('result/reynolds_et_al_2021/solo/solo.csv')

In [ ]:
adata.write('data/object/reynolds_et_al_2021/solo/COMB/adata.h5ad')